In [1]:
import os
import numpy as np
import mne

from scipy.io import loadmat

from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
DATA_FOLDER = "../data/BCI_IV_2a"
LABEL_FOLDER = "../data/true_labels"

print("GDF folder:", os.path.abspath(DATA_FOLDER))
print("Label folder:", os.path.abspath(LABEL_FOLDER))

GDF folder: C:\Users\chahi\Desktop\EEG-Project\data\BCI_IV_2a
Label folder: C:\Users\chahi\Desktop\EEG-Project\data\true_labels


In [3]:
# Fixed OUTPUT labels: 1=LEFT, 2=RIGHT, 3=FOOT, 4=TONGUE.
# IMPORTANT: MNE assigns per-file numeric codes to annotation
# descriptions based on which descriptions appear in that specific
# file and in what order -- the same description (e.g. "769") can get
# a DIFFERENT numeric code in different subjects' files (confirmed for
# A04T in A04T_Investigation.ipynb). We always look codes up by
# description string per file, then translate to these fixed labels.
DESC_TO_LABEL = {
    "769": 1,   # LEFT
    "770": 2,   # RIGHT
    "771": 3,   # FOOT
    "772": 4,   # TONGUE
}


def load_training(subject):

    path = os.path.join(
        DATA_FOLDER,
        subject + "T.gdf"
    )

    raw = mne.io.read_raw_gdf(
        path,
        preload=True,
        verbose=False
    )

    raw.pick("eeg")

    events, event_dict = mne.events_from_annotations(
        raw,
        verbose=False
    )

    # Build this subject's event_id using description strings
    available = {
        desc: event_dict[desc]
        for desc in DESC_TO_LABEL
        if desc in event_dict
    }

    if len(available) != 4:
        missing = set(DESC_TO_LABEL) - set(available)
        print(f"WARNING: {subject} missing motor-imagery events: {missing}")

    epochs = mne.Epochs(
        raw,
        events,
        event_id=available,
        tmin=0,
        tmax=4,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()
    X = X[:, :22, :]

    # Remap this subject's per-file codes to the fixed output labels
    code_to_label = {
        event_dict[desc]: DESC_TO_LABEL[desc]
        for desc in available
    }

    y = np.array([
        code_to_label[code]
        for code in epochs.events[:, -1]
    ])

    return X, y


In [4]:
def load_testing(subject):

    path = os.path.join(
        DATA_FOLDER,
        subject + "E.gdf"
    )

    raw = mne.io.read_raw_gdf(
        path,
        preload=True,
        verbose=False
    )

    raw.pick("eeg")

    events, event_dict = mne.events_from_annotations(
        raw,
        verbose=False
    )

    # "783" = evaluation/unknown-cue trial marker. Look its per-file
    # code up by description rather than assuming it is always 7 --
    # the same numeric-code drift confirmed for A04T's training file
    # can happen here too.
    if "783" not in event_dict:
        raise ValueError(
            f"{subject}E: '783' trial marker not found. "
            f"Available events: {event_dict}"
        )

    trial_code = event_dict["783"]

    target_events = events[
        events[:, 2] == trial_code
    ]

    if len(target_events) != 288:
        raise ValueError(
            f"{subject}E: expected 288 trials, "
            f"found {len(target_events)}"
        )

    epochs = mne.Epochs(
        raw,
        target_events,
        event_id={"MI": trial_code},
        tmin=0,
        tmax=4,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()
    X = X[:, :22, :]

    # True labels
    mat_path = os.path.join(
        LABEL_FOLDER,
        subject + "E.mat"
    )

    mat = loadmat(mat_path)

    y = np.asarray(
        mat["classlabel"]
    ).flatten().astype(int)

    if len(y) != 288:
        raise ValueError(
            f"{subject}E: expected 288 labels, "
            f"found {len(y)}"
        )

    return X, y


In [5]:
def filter_data(X):

    X_filtered = np.empty_like(X)

    for i in range(len(X)):

        X_filtered[i] = mne.filter.filter_data(
            X[i],
            sfreq=250,
            l_freq=8,
            h_freq=30,
            verbose=False
        )

    return X_filtered

In [6]:
def run_adaptation(
    X_train,
    y_train,
    X_test,
    y_test,
    adaptation_percentage
):

    # Shuffle E-session trials reproducibly
    rng = np.random.RandomState(42)

    indices = rng.permutation(len(X_test))

    n_adapt = int(
        len(X_test) * adaptation_percentage
    )

    adapt_idx = indices[:n_adapt]
    eval_idx = indices[n_adapt:]

    X_adapt = X_test[adapt_idx]
    y_adapt = y_test[adapt_idx]

    X_eval = X_test[eval_idx]
    y_eval = y_test[eval_idx]

    # Combine original training data
    # with labeled adaptation trials
    X_combined = np.concatenate(
        [X_train, X_adapt],
        axis=0
    )

    y_combined = np.concatenate(
        [y_train, y_adapt],
        axis=0
    )

    # CSP is fitted ONLY using training/adaptation data
    csp = CSP(
        n_components=12,
        reg=None,
        log=True,
        norm_trace=False
    )

    X_combined_csp = csp.fit_transform(
        X_combined,
        y_combined
    )

    X_eval_csp = csp.transform(
        X_eval
    )

    # LDA
    lda = LinearDiscriminantAnalysis()

    lda.fit(
        X_combined_csp,
        y_combined
    )

    y_pred = lda.predict(
        X_eval_csp
    )

    accuracy = accuracy_score(
        y_eval,
        y_pred
    )

    macro_f1 = f1_score(
        y_eval,
        y_pred,
        average="macro"
    )

    return accuracy, macro_f1, y_eval, y_pred

In [7]:
subject = "A01"

print("=" * 60)
print("SUBJECT:", subject)
print("=" * 60)

X_train, y_train = load_training(subject)

X_test, y_test = load_testing(subject)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

print(
    "Training labels:",
    np.unique(y_train, return_counts=True)
)

print(
    "Testing labels:",
    np.unique(y_test, return_counts=True)
)

SUBJECT: A01


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Training: (288, 22, 1001)
Testing : (288, 22, 1001)
Training labels: (array([1, 2, 3, 4]), array([72, 72, 72, 72]))
Testing labels: (array([1, 2, 3, 4]), array([72, 72, 72, 72]))


In [8]:
X_train = filter_data(X_train)
X_test = filter_data(X_test)

print("Filtered training:", X_train.shape)
print("Filtered testing :", X_test.shape)

Filtered training: (288, 22, 1001)
Filtered testing : (288, 22, 1001)


In [9]:
adaptation_levels = [
    0.00,
    0.05,
    0.10,
    0.20
]

results = []

for percentage in adaptation_levels:

    accuracy, macro_f1, y_true, y_pred = run_adaptation(
        X_train,
        y_train,
        X_test,
        y_test,
        percentage
    )

    results.append(
        (
            percentage,
            accuracy,
            macro_f1
        )
    )

    print(
        f"Adaptation {percentage*100:.0f}%:"
    )

    print(
        f"Accuracy: {accuracy*100:.2f}%"
    )

    print(
        f"Macro F1: {macro_f1:.4f}"
    )

    print("-" * 40)

Computing rank from data with rank=None
    Using tolerance 5.8e-05 (2.2e-16 eps * 22 dim * 1.2e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
Adaptation 0%:
Accuracy: 69.44%
Macro F1: 0.6865
----------------------------------------
Computing rank from data with rank=None
    Using tolerance 6e-05 (2.2e-16 eps * 22 dim * 1.2e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL


In [10]:
print("=" * 60)
print("A01 SUBJECT ADAPTATION RESULTS")
print("=" * 60)

for percentage, accuracy, macro_f1 in results:

    print(
        f"{percentage*100:>5.0f}% adaptation"
        f" → Accuracy: {accuracy*100:.2f}%"
        f" | Macro F1: {macro_f1:.4f}"
    )

A01 SUBJECT ADAPTATION RESULTS
    0% adaptation → Accuracy: 69.44% | Macro F1: 0.6865
    5% adaptation → Accuracy: 68.25% | Macro F1: 0.6742
   10% adaptation → Accuracy: 69.23% | Macro F1: 0.6875
   20% adaptation → Accuracy: 72.73% | Macro F1: 0.7268


In [11]:
import numpy as np

subjects = [
    "A01", "A02", "A03",
    "A04", "A05", "A06",
    "A07", "A08", "A09"
]

adaptation_levels = [0.00, 0.05, 0.10, 0.20]

results = {
    level: {
        "accuracy": [],
        "f1": []
    }
    for level in adaptation_levels
}


for subject in subjects:

    print("\n" + "=" * 60)
    print(f"SUBJECT: {subject}")
    print("=" * 60)

    # -----------------------------
    # Load training session
    # -----------------------------
    X_train, y_train = load_training(subject)

    # -----------------------------
    # Load evaluation session
    # -----------------------------
    X_test, y_test = load_testing(subject)

    print("Training:", X_train.shape)
    print("Testing :", X_test.shape)

    # -----------------------------
    # Test each adaptation level
    # -----------------------------
    for level in adaptation_levels:

        accuracy, macro_f1, y_true, y_pred = run_adaptation(
            X_train,
            y_train,
            X_test,
            y_test,
            level
        )

        results[level]["accuracy"].append(accuracy)
        results[level]["f1"].append(macro_f1)

        print(
            f"{level*100:>5.0f}% adaptation → "
            f"Accuracy: {accuracy*100:.2f}% | "
            f"Macro F1: {macro_f1:.4f}"
        )


SUBJECT: A01


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Training: (288, 22, 1001)
Testing : (288, 22, 1001)
Computing rank from data with rank=None
    Using tolerance 0.00012 (2.2e-16 eps * 22 dim * 2.4e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
    0% adaptation → Accuracy: 58.33% | Macro F1: 0.5578
Computing rank from data with rank=None
    Using tolerance 0.00012 (2.2e-16 eps * 22 dim * 2.4e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covaria

C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Training: (288, 22, 1001)
Testing : (288, 22, 1001)
Computing rank from data with rank=None
    Using tolerance 0.0001 (2.2e-16 eps * 22 dim * 2e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
    0% adaptation → Accuracy: 39.93% | Macro F1: 0.3539
Computing rank from data with rank=None
    Using tolerance 0.0001 (2.2e-16 eps * 22 dim * 2.1e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance 

...

Training: (288, 22, 1001)
Testing : (288, 22, 1001)
Computing rank from data with rank=None
    Using tolerance 9.6e-05 (2.2e-16 eps * 22 dim * 2e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
    0% adaptation → Accuracy: 55.56% | Macro F1: 0.5562
Computing rank from data with rank=None
    Using tolerance 9.8e-05 (2.2e-16 eps * 22 dim * 2e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance 

C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Training: (288, 22, 1001)
Testing : (288, 22, 1001)
Computing rank from data with rank=None
    Using tolerance 0.00015 (2.2e-16 eps * 22 dim * 3e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
    0% adaptation → Accuracy: 63.89% | Macro F1: 0.6349
Computing rank from data with rank=None
    Using tolerance 0.00015 (2.2e-16 eps * 22 dim * 3e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance 

C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Training: (288, 22, 1001)
Testing : (288, 22, 1001)
Computing rank from data with rank=None
    Using tolerance 0.00015 (2.2e-16 eps * 22 dim * 3e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
    0% adaptation → Accuracy: 66.67% | Macro F1: 0.6307
Computing rank from data with rank=None
    Using tolerance 0.00015 (2.2e-16 eps * 22 dim * 3.1e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covarianc

In [12]:
print("\n")
print("=" * 70)
print("FINAL SUBJECT ADAPTATION RESULTS")
print("=" * 70)

for level in adaptation_levels:

    acc = np.array(results[level]["accuracy"])
    f1 = np.array(results[level]["f1"])

    print(
        f"{level*100:>5.0f}% adaptation → "
        f"{acc.mean()*100:.2f}% ± {acc.std()*100:.2f}% | "
        f"F1: {f1.mean():.4f} ± {f1.std():.4f}"
    )



FINAL SUBJECT ADAPTATION RESULTS
    0% adaptation → 50.73% ± 14.82% | F1: 0.4737 ± 0.1822
    5% adaptation → 50.77% ± 14.69% | F1: 0.4682 ± 0.1656
   10% adaptation → 54.19% ± 13.63% | F1: 0.5246 ± 0.1431
   20% adaptation → 58.30% ± 11.98% | F1: 0.5704 ± 0.1276


In [13]:
print("\n")
print("=" * 70)
print("SUBJECT-BY-SUBJECT ACCURACY")
print("=" * 70)

print(
    f"{'Subject':<10}"
    f"{'0%':>10}"
    f"{'5%':>10}"
    f"{'10%':>10}"
    f"{'20%':>10}"
)

print("-" * 50)

for i, subject in enumerate(subjects):

    print(
        f"{subject:<10}"
        f"{results[0.00]['accuracy'][i]*100:>9.2f}%"
        f"{results[0.05]['accuracy'][i]*100:>9.2f}%"
        f"{results[0.10]['accuracy'][i]*100:>9.2f}%"
        f"{results[0.20]['accuracy'][i]*100:>9.2f}%"
    )



SUBJECT-BY-SUBJECT ACCURACY
Subject           0%        5%       10%       20%
--------------------------------------------------
A01           58.33%    54.74%    55.00%    55.41%
A02           39.93%    43.43%    47.31%    52.38%
A03           69.79%    70.44%    70.00%    74.46%
A04           43.06%    36.50%    45.00%    47.19%
A05           25.00%    29.20%    31.92%    42.42%
A06           34.38%    37.59%    40.00%    45.89%
A07           55.56%    48.18%    55.77%    60.17%
A08           63.89%    70.80%    71.92%    73.59%
A09           66.67%    66.06%    70.77%    73.16%


In [14]:
print("\n")
print("=" * 70)
print("IMPROVEMENT OVER 0% ADAPTATION")
print("=" * 70)

baseline = np.mean(results[0.00]["accuracy"])

for level in [0.05, 0.10, 0.20]:

    adapted = np.mean(results[level]["accuracy"])

    improvement = (adapted - baseline) * 100

    print(
        f"{level*100:>5.0f}% adaptation → "
        f"{improvement:+.2f} percentage points"
    )



IMPROVEMENT OVER 0% ADAPTATION
    5% adaptation → +0.04 percentage points
   10% adaptation → +3.46 percentage points
   20% adaptation → +7.56 percentage points
